# Lab 06: Landmark Recognition Using the SIFT Algorithm

**Objective:** Build a computer vision system that recognizes famous landmarks
from query images using SIFT (Scale-Invariant Feature Transform) feature matching.

**Pipeline:**
Query Image -> SIFT Keypoints & Descriptors -> BFMatcher + KNN ->
Lowe's Ratio Test -> Recognition Decision -> Homography Localization

**Tasks:**
1. Build landmark image database (5 landmarks, multiple views)
2. SIFT feature extraction on reference images
3. Query image processing
4. Feature matching with BFMatcher + KNN + Lowe's ratio test
5. Landmark localization via homography (RANSAC)
6. Performance evaluation under various conditions

## 1. Setup and Installation

In [1]:
import subprocess, sys, os

# Uninstall default OpenCV (conflicts with SIFT)
for pkg in ["opencv-python", "opencv-python-headless"]:
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", pkg],
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Install OpenCV with SIFT support
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "opencv-contrib-python-headless"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("OpenCV version:", cv2.__version__)
sift_test = cv2.SIFT_create()
print("SIFT available: True")
print("Setup complete.")

OpenCV version: 5.0.0
SIFT available: True
Setup complete.


## 2. Build Landmark Image Database (Task 1)

Download real landmark photos from Wikimedia Commons using
the official API to get valid thumbnail URLs.

In [2]:
import urllib.request, ssl, os, json, time
import cv2
import numpy as np

DB_DIR = "landmark_database"
os.makedirs(DB_DIR, exist_ok=True)

# Wikimedia Commons file names for 5 famous landmarks, 3 images each
LANDMARK_FILES = {
    "Eiffel_Tower": [
        "File:Tour Eiffel Wikimedia Commons (cropped).jpg",
        "File:Tour Eiffel Wikimedia Commons.jpg",
        "File:Eiffel Tower 20051010.jpg",
    ],
    "Taj_Mahal": [
        "File:Taj Mahal, Agra, India edit3.jpg",
        "File:Taj Mahal (Edited).jpeg",
        "File:Taj Mahal in March 2004.jpg",
    ],
    "Great_Wall": [
        "File:The Great Wall of China at Jinshanling-edit.jpg",
        "File:The Great Wall of China at Jinshanling.jpg",
        "File:20090529 Great Wall 8185.jpg",
    ],
    "Statue_of_Liberty": [
        "File:Statue of Liberty 7.jpg",
        "File:Lady Liberty under a blue sky (cropped).jpg",
        "File:Statue of Liberty, NY.jpg",
    ],
    "Colosseum": [
        "File:Colosseo 2020.jpg",
        "File:Colosseum in Rome, Italy - April 2007.jpg",
        "File:Colosseum in Rome-April 2007-1- copie 2B.jpg",
    ],
}

def get_wiki_thumb_url(file_title, width=400):
    'Use Wikimedia API to get a valid thumbnail URL.'
    import urllib.parse
    api = "https://en.wikipedia.org/w/api.php"
    params = urllib.parse.urlencode({
        "action": "query",
        "titles": file_title,
        "prop": "imageinfo",
        "iiprop": "url",
        "iiurlwidth": width,
        "format": "json",
    })
    url = f"{api}?{params}"
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    req = urllib.request.Request(url, headers={"User-Agent": "LandmarkRecognitionLab/1.0"})
    resp = urllib.request.urlopen(req, timeout=20, context=ctx)
    data = json.loads(resp.read().decode())
    pages = data.get("query", {}).get("pages", {})
    for page_id, page in pages.items():
        if "imageinfo" in page:
            info = page["imageinfo"][0]
            return info.get("thumburl", info.get("url"))
    return None

def download_url(url, path, retries=3):
    'Download a URL to a file with retries.'
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers={
                "User-Agent": "LandmarkRecognitionLab/1.0"
            })
            data = urllib.request.urlopen(req, timeout=30, context=ctx).read()
            if len(data) < 500:
                continue
            with open(path, "wb") as f:
                f.write(data)
            img = cv2.imread(path)
            if img is not None and img.shape[0] > 10:
                return True
            if os.path.exists(path):
                os.remove(path)
        except Exception:
            time.sleep(1)
    return False

def make_augmented(base_img, variant):
    'Create augmented variant from a base image.'
    h, w = base_img.shape[:2]
    if variant == 0:
        m = h // 10
        return base_img[m:h-m, m:w-m].copy()
    elif variant == 1:
        M = cv2.getRotationMatrix2D((w/2, h/2), 10, 1.0)
        rot = cv2.warpAffine(base_img, M, (w, h), borderMode=cv2.BORDER_REFLECT)
        return np.clip(rot.astype(np.int16) + 25, 0, 255).astype(np.uint8)
    else:
        return cv2.flip(base_img, 1)

def make_synthetic(landmark_name, idx):
    'Last resort: create a textured synthetic image with the landmark name.'
    rng = np.random.RandomState((idx + 1) * 12345 + len(landmark_name) * 67)
    img = rng.randint(30, 200, (400, 400, 3), dtype=np.uint8)
    img = cv2.GaussianBlur(img, (7, 7), 0)
    label = landmark_name.replace("_", " ")
    cv2.putText(img, label, (30, 200), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(img, f"View {idx+1}", (30, 240), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)
    return img

# ---- Download all images ----
print("Downloading real landmark images from Wikimedia Commons...")
print("Using the Wikimedia API to get valid thumbnail URLs\n")

for landmark, files in LANDMARK_FILES.items():
    lm_dir = os.path.join(DB_DIR, landmark)
    os.makedirs(lm_dir, exist_ok=True)
    downloaded_paths = []

    for i, wiki_file in enumerate(files):
        fname = f"{landmark}_{i+1}.jpg"
        fpath = os.path.join(lm_dir, fname)

        # Skip if already downloaded
        if os.path.exists(fpath):
            test = cv2.imread(fpath)
            if test is not None and test.shape[0] > 10:
                print(f"  [cached] {landmark}/{fname}")
                downloaded_paths.append(fpath)
                continue

        print(f"  Downloading {landmark}/{fname}...", end=" ", flush=True)

        # Method 1: Use Wikimedia API
        success = False
        try:
            thumb_url = get_wiki_thumb_url(wiki_file, width=400)
            if thumb_url:
                success = download_url(thumb_url, fpath)
        except Exception:
            pass

        if success:
            print("OK")
            downloaded_paths.append(fpath)
        else:
            print("FAILED")

    # Fill missing with augmentation from successful downloads
    existing = [f for f in os.listdir(lm_dir)
                if f.endswith((".jpg", ".png"))
                and cv2.imread(os.path.join(lm_dir, f)) is not None]

    if 0 < len(existing) < 3:
        base = cv2.imread(os.path.join(lm_dir, existing[0]))
        for v in range(3 - len(existing)):
            aug_name = f"{landmark}_aug{v+1}.jpg"
            aug_path = os.path.join(lm_dir, aug_name)
            aug = make_augmented(base, v)
            cv2.imwrite(aug_path, aug)
            print(f"  [augmented] {landmark}/{aug_name}")

    # Last resort: synthetic placeholders
    existing2 = [f for f in os.listdir(lm_dir)
                 if f.endswith((".jpg", ".png"))
                 and cv2.imread(os.path.join(lm_dir, f)) is not None]
    if len(existing2) == 0:
        print(f"  [!] All downloads failed for {landmark} - creating synthetic images")
        for v in range(3):
            syn = make_synthetic(landmark, v)
            cv2.imwrite(os.path.join(lm_dir, f"{landmark}_syn{v+1}.jpg"), syn)

# ---- Summary ----
print("\n" + "=" * 50)
print("LANDMARK DATABASE SUMMARY")
print("=" * 50)
landmark_names = sorted([d for d in os.listdir(DB_DIR)
                         if os.path.isdir(os.path.join(DB_DIR, d))])
total = 0
for lm in landmark_names:
    lm_dir = os.path.join(DB_DIR, lm)
    count = len([f for f in os.listdir(lm_dir) if f.endswith((".jpg", ".png"))])
    total += count
    print(f"  {lm}: {count} images")
print(f"\nTotal: {total} reference images across {len(landmark_names)} landmarks")
print("=" * 50)

Using the Wikimedia API to get valid thumbnail URLs

  [augmented] Statue_of_Liberty/Statue_of_Liberty_aug1.jpg
  [augmented] Statue_of_Liberty/Statue_of_Liberty_aug2.jpg
  [!] All downloads failed for Colosseum - creating synthetic images

LANDMARK DATABASE SUMMARY
  Colosseum: 3 images
  Eiffel_Tower: 3 images
  Great_Wall: 3 images
  Statue_of_Liberty: 3 images
  Taj_Mahal: 3 images

Total: 15 reference images across 5 landmarks


In [3]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt

DB_DIR = "landmark_database"
OUT_DIR = "lab06_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

landmark_names = sorted([d for d in os.listdir(DB_DIR)
                         if os.path.isdir(os.path.join(DB_DIR, d))])
n_landmarks = len(landmark_names)

fig, axes = plt.subplots(n_landmarks, 3, figsize=(15, n_landmarks * 3))
if n_landmarks == 1:
    axes = axes.reshape(1, -1)

for i, lm in enumerate(landmark_names):
    lm_dir = os.path.join(DB_DIR, lm)
    imgs = sorted([f for f in os.listdir(lm_dir) if f.endswith((".jpg", ".png"))])
    for j in range(3):
        ax = axes[i, j] if n_landmarks > 1 else axes[0, j]
        if j < len(imgs):
            img = cv2.imread(os.path.join(lm_dir, imgs[j]))
            if img is not None:
                ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                if j == 0:
                    ax.set_ylabel(lm.replace("_", " "), fontsize=11, fontweight="bold")
                ax.set_title(f"View {j+1}", fontsize=9)
        ax.axis("off")

plt.suptitle("Task 1: Landmark Image Database (5 landmarks x 3 views)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_database.png", dpi=150, bbox_inches="tight")
plt.show()
print("Database visualization saved.")

Database visualization saved.


## 3. SIFT Feature Extraction (Task 2)

**SIFT** detects distinctive keypoints that are invariant to:
- **Scale** - features detected at multiple zoom levels
- **Rotation** - each keypoint's orientation is normalized
- **Illumination** - uses gradient orientations, not raw intensities

Each keypoint produces a **128-dimensional descriptor** used for matching.

In [4]:
import os, cv2
import numpy as np

DB_DIR = "landmark_database"
landmark_names = sorted([d for d in os.listdir(DB_DIR)
                         if os.path.isdir(os.path.join(DB_DIR, d))])
n_landmarks = len(landmark_names)

sift = cv2.SIFT_create()

# Build database: {landmark: [{"kp", "des", "img", "gray", "file"}, ...]}
database = {}

print("Extracting SIFT features from all reference images...")
print(f"{'Landmark':<25} {'Image':<30} {'Keypoints':<12} {'Descriptor Shape'}")
print("-" * 85)

for lm in landmark_names:
    lm_dir = os.path.join(DB_DIR, lm)
    database[lm] = []
    files = sorted([f for f in os.listdir(lm_dir) if f.endswith((".jpg", ".png"))])
    for fname in files:
        img = cv2.imread(os.path.join(lm_dir, fname))
        if img is None:
            continue
        img = cv2.resize(img, (400, 400))
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        kp, des = sift.detectAndCompute(gray, None)
        if des is None:
            continue
        database[lm].append({
            "kp": kp, "des": des,
            "img": img, "gray": gray, "file": fname
        })
        print(f"  {lm:<23} {fname:<30} {len(kp):<12} {des.shape}")

total_imgs = sum(len(v) for v in database.values())
print(f"\nDatabase ready: {len(database)} landmarks, {total_imgs} images total")

Extracting SIFT features from all reference images...
Landmark                  Image                          Keypoints    Descriptor Shape
-------------------------------------------------------------------------------------
  Colosseum               Colosseum_syn1.jpg             93           (93, 128)
  Colosseum               Colosseum_syn2.jpg             92           (92, 128)
  Colosseum               Colosseum_syn3.jpg             91           (91, 128)
  Eiffel_Tower            Eiffel_Tower_1.jpg             563          (563, 128)
  Eiffel_Tower            Eiffel_Tower_2.jpg             459          (459, 128)
  Eiffel_Tower            Eiffel_Tower_3.jpg             496          (496, 128)
  Great_Wall              Great_Wall_1.jpg               1309         (1309, 128)
  Great_Wall              Great_Wall_2.jpg               1417         (1417, 128)
  Great_Wall              Great_Wall_3.jpg               1463         (1463, 128)
  Statue_of_Liberty       Statue_of_Liberty_

In [5]:
import matplotlib.pyplot as plt

OUT_DIR = "lab06_outputs"

fig, axes = plt.subplots(2, n_landmarks, figsize=(4 * n_landmarks, 8))

for i, lm in enumerate(landmark_names):
    refs = database.get(lm, [])
    if len(refs) == 0:
        axes[0, i].set_title(lm + " (no data)", color="red")
        axes[0, i].axis("off")
        axes[1, i].axis("off")
        continue

    ref = refs[0]
    img_kp = cv2.drawKeypoints(ref["gray"], ref["kp"], None,
                                color=(0, 255, 0),
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)

    axes[0, i].imshow(cv2.cvtColor(ref["img"], cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(lm.replace("_", " "), fontsize=10, fontweight="bold")
    axes[0, i].axis("off")

    axes[1, i].imshow(cv2.cvtColor(img_kp, cv2.COLOR_BGR2RGB))
    axes[1, i].set_title(f"{len(ref['kp'])} keypoints", fontsize=9)
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("SIFT Keypoints", fontsize=10)
plt.suptitle("Task 2: SIFT Feature Extraction", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_sift_keypoints.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Query Image Processing (Task 3)

We take one image from the database, treat it as an unknown query,
and extract its SIFT keypoints and descriptors.

In [6]:
import matplotlib.pyplot as plt

OUT_DIR = "lab06_outputs"

# Pick the last view of the first landmark as the query
query_landmark = landmark_names[0]
query_ref = database[query_landmark][-1]
query_img = query_ref["img"].copy()
query_gray = query_ref["gray"].copy()

query_kp, query_des = sift.detectAndCompute(query_gray, None)

print(f"Query image: {query_ref['file']} (from {query_landmark})")
print(f"Keypoints detected: {len(query_kp)}")
print(f"Descriptor shape: {query_des.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Query Image (Unknown Landmark)", fontsize=12)
axes[0].axis("off")

kp_vis = cv2.drawKeypoints(query_gray, query_kp, None, color=(0, 0, 255),
                            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
axes[1].imshow(cv2.cvtColor(kp_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f"SIFT Keypoints ({len(query_kp)} detected)", fontsize=12)
axes[1].axis("off")

plt.suptitle("Task 3: Query Image Processing", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_query_processing.png", dpi=150, bbox_inches="tight")
plt.show()

Query image: Colosseum_syn3.jpg (from Colosseum)
Keypoints detected: 91
Descriptor shape: (91, 128)


## 5. Feature Matching and Landmark Recognition (Task 4)

**Matching pipeline:**
1. **BFMatcher** with L2 (Euclidean) distance
2. **KNN (k=2)** to find two nearest neighbors per descriptor
3. **Lowe's Ratio Test** (ratio < 0.75) to filter ambiguous matches
4. **Threshold** to reject if too few reliable matches

In [7]:
# ---- TASK 4: Match query against database ----

MIN_MATCH_THRESHOLD = 15  # Minimum reliable matches for recognition
RATIO_THRESH = 0.75       # Lowe's ratio test threshold

def match_landmark(q_des, db, ratio=0.75):
    """Match query descriptors against all reference images."""
    bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=False)
    results = []

    for lm, refs in db.items():
        best_count = 0
        best_good = []
        best_ref = None

        for ref in refs:
            if ref["des"] is None or q_des is None:
                continue
            try:
                raw = bf.knnMatch(q_des, ref["des"], k=2)
            except cv2.error:
                continue

            # Lowe's ratio test
            good = []
            for pair in raw:
                if len(pair) == 2:
                    m, n = pair
                    if m.distance < ratio * n.distance:
                        good.append(m)

            if len(good) > best_count:
                best_count = len(good)
                best_good = good
                best_ref = ref

        results.append({
            "landmark": lm,
            "matches": best_count,
            "good_matches": best_good,
            "ref": best_ref,
        })

    results.sort(key=lambda x: x["matches"], reverse=True)
    return results

# Run matching
results = match_landmark(query_des, database, ratio=RATIO_THRESH)

print("Task 4: Landmark Recognition Results")
print("=" * 65)
print(f"{'Rank':<6} {'Landmark':<25} {'Reliable Matches':<18} {'Decision'}")
print("-" * 65)

for rank, r in enumerate(results, 1):
    if rank == 1 and r["matches"] >= MIN_MATCH_THRESHOLD:
        decision = ">>> RECOGNIZED <<<"
    else:
        decision = "Rejected"
    print(f"  {rank:<4} {r['landmark']:<25} {r['matches']:<18} {decision}")

top = results[0]
print(f"\n{'=' * 65}")
if top["matches"] >= MIN_MATCH_THRESHOLD:
    print(f"PREDICTION: {top['landmark'].replace('_', ' ')}")
    print(f"Matches: {top['matches']} (threshold: {MIN_MATCH_THRESHOLD})")
else:
    print(f"UNKNOWN LANDMARK (best: {top['matches']} < threshold {MIN_MATCH_THRESHOLD})")
print("=" * 65)

Task 4: Landmark Recognition Results
Rank   Landmark                  Reliable Matches   Decision
-----------------------------------------------------------------
  1    Colosseum                 91                 >>> RECOGNIZED <<<
  2    Taj_Mahal                 3                  Rejected
  3    Statue_of_Liberty         2                  Rejected
  4    Eiffel_Tower              1                  Rejected
  5    Great_Wall                1                  Rejected

PREDICTION: Colosseum
Matches: 91 (threshold: 15)


In [8]:
import matplotlib.pyplot as plt

OUT_DIR = "lab06_outputs"

# Draw feature matches
top = results[0]
if top["ref"] is not None and len(top["good_matches"]) > 0:
    match_img = cv2.drawMatches(
        query_img, query_kp,
        top["ref"]["img"], top["ref"]["kp"],
        top["good_matches"][:50], None,
        matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)

    plt.figure(figsize=(18, 6))
    plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Feature Matching: Query vs {top['landmark'].replace('_', ' ')} "
              f"({top['matches']} matches)", fontsize=13)
    plt.axis("off")
    plt.savefig(f"{OUT_DIR}/04_matching.png", dpi=150, bbox_inches="tight")
    plt.show()

# Bar chart of match counts
fig, ax = plt.subplots(figsize=(10, 5))
names = [r["landmark"].replace("_", " ") for r in results]
counts = [r["matches"] for r in results]
colors = ["#4CAF50" if c >= MIN_MATCH_THRESHOLD else "#EF5350" for c in counts]
bars = ax.barh(range(len(names)), counts, color=colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names)
ax.axvline(MIN_MATCH_THRESHOLD, color="black", linestyle="--", linewidth=1,
           label=f"Threshold ({MIN_MATCH_THRESHOLD})")
ax.set_xlabel("Reliable Matches (after Lowe's ratio test)")
ax.set_title("Match Count per Landmark")
ax.legend()
ax.invert_yaxis()
for i, v in enumerate(counts):
    ax.text(v + 1, i, str(v), va="center", fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_match_chart.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Landmark Localization (Task 5)

With enough matches, we estimate a **homography** using **RANSAC**
to project the reference boundary onto the query image,
drawing a bounding polygon around the recognized landmark.

In [9]:
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR = "lab06_outputs"

def localize_landmark(q_img, q_kp, ref, good_matches, min_inliers=8):
    """Estimate homography via RANSAC and draw landmark boundary."""
    result = q_img.copy()

    if len(good_matches) < 4:
        return result, False, 0

    src_pts = np.float32([ref["kp"][m.trainIdx].pt
                          for m in good_matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([q_kp[m.queryIdx].pt
                          for m in good_matches]).reshape(-1, 1, 2)

    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)

    if H is None:
        return result, False, 0

    inliers = int(mask.sum()) if mask is not None else 0

    if inliers < min_inliers:
        return result, False, inliers

    h, w = ref["gray"].shape
    corners = np.float32([[0,0],[w,0],[w,h],[0,h]]).reshape(-1, 1, 2)
    projected = cv2.perspectiveTransform(corners, H)

    # Validate polygon area
    area = cv2.contourArea(projected.reshape(4, 2).astype(np.float32))
    q_area = q_img.shape[0] * q_img.shape[1]
    if area < 100 or area > q_area * 0.95:
        return result, False, inliers

    cv2.polylines(result, [np.int32(projected)], True, (0, 255, 0), 3)
    top_pt = tuple(np.int32(projected)[0][0])
    cv2.putText(result, ref["file"].split("_")[0],
                (top_pt[0], max(top_pt[1] - 10, 25)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    return result, True, inliers

# Localize the top match
top = results[0]
print(f"Localizing: {top['landmark']}")
loc_img, loc_ok, loc_inliers = localize_landmark(
    query_img, query_kp, top["ref"], top["good_matches"])
print(f"  RANSAC inliers: {loc_inliers}")
print(f"  Localization: {'SUCCESS' if loc_ok else 'FAILED'}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(query_img, cv2.COLOR_BGR2RGB))
axes[0].set_title("Query Image", fontsize=12)
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(loc_img, cv2.COLOR_BGR2RGB))
status = "Landmark Located" if loc_ok else "Localization Failed"
axes[1].set_title(f"Task 5: {status}\n{top['landmark'].replace('_',' ')} "
                  f"({top['matches']} matches, {loc_inliers} inliers)", fontsize=11)
axes[1].axis("off")

plt.suptitle("Task 5: Landmark Localization via Homography + RANSAC", fontsize=14)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_localization.png", dpi=150, bbox_inches="tight")
plt.show()

Localizing: Colosseum
  RANSAC inliers: 91
  Localization: FAILED


## 7. Performance Evaluation (Task 6)

Testing system robustness under:
1. **Original** -- unmodified query
2. **Rotated** -- 30 degree rotation
3. **Resized** -- 50% scale
4. **Darkened** -- brightness reduced by 40%
5. **Different viewpoint** -- another view of same landmark
6. **Unknown** -- random image not in database

In [10]:
import time
import matplotlib.pyplot as plt
import numpy as np

OUT_DIR = "lab06_outputs"

def create_test_conditions(img, db, q_landmark):
    """Generate test images under various conditions."""
    h, w = img.shape[:2]
    conds = {}

    # 1. Original
    conds["Original"] = img.copy()

    # 2. Rotated 30 degrees
    M = cv2.getRotationMatrix2D((w/2, h/2), 30, 1.0)
    conds["Rotated 30deg"] = cv2.warpAffine(img, M, (w, h),
                                             borderMode=cv2.BORDER_REFLECT)

    # 3. Resized to 50%
    small = cv2.resize(img, (w//2, h//2))
    padded = np.zeros_like(img)
    padded[:h//2, :w//2] = small
    conds["Resized 50%"] = padded

    # 4. Darkened
    dark = np.clip(img.astype(np.float32) * 0.6, 0, 255).astype(np.uint8)
    conds["Dark -40%"] = dark

    # 5. Different viewpoint
    views = db.get(q_landmark, [])
    if len(views) > 1:
        conds["Diff Viewpoint"] = views[0]["img"].copy()
    else:
        conds["Diff Viewpoint"] = cv2.flip(img, 1)

    # 6. Unknown image (not in database)
    rng = np.random.RandomState(999)
    noise = rng.randint(40, 200, img.shape, dtype=np.uint8)
    conds["Unknown"] = cv2.GaussianBlur(noise, (21, 21), 0)

    return conds

test_conds = create_test_conditions(query_img, database, query_landmark)

print("Task 6: Performance Evaluation")
print("=" * 100)
print(f"{'Condition':<18} {'Predicted':<22} {'Keypoints':<12} {'Matches':<10} {'Decision':<14} {'Time (ms)'}")
print("-" * 100)

eval_results = []
for cname, cimg in test_conds.items():
    t0 = time.time()
    cgray = cv2.cvtColor(cimg, cv2.COLOR_BGR2GRAY) if len(cimg.shape) == 3 else cimg
    kp, des = sift.detectAndCompute(cgray, None)

    if des is None or len(kp) < 2:
        elapsed = (time.time() - t0) * 1000
        eval_results.append({
            "cond": cname, "pred": "N/A", "kp": 0,
            "matches": 0, "decision": "REJECTED", "ms": round(elapsed, 1)
        })
        print(f"  {cname:<16} {'N/A':<22} {0:<12} {0:<10} {'REJECTED':<14} {elapsed:.1f}")
        continue

    res = match_landmark(des, database, ratio=RATIO_THRESH)
    elapsed = (time.time() - t0) * 1000
    top_r = res[0]

    if top_r["matches"] >= MIN_MATCH_THRESHOLD:
        decision = "RECOGNIZED"
        pred = top_r["landmark"].replace("_", " ")
    else:
        decision = "REJECTED"
        pred = "Unknown"

    eval_results.append({
        "cond": cname, "pred": pred, "kp": len(kp),
        "matches": top_r["matches"], "decision": decision, "ms": round(elapsed, 1)
    })
    print(f"  {cname:<16} {pred:<22} {len(kp):<12} {top_r['matches']:<10} {decision:<14} {elapsed:.1f}")

print("=" * 100)

known_ok = sum(1 for r in eval_results[:5] if r["decision"] == "RECOGNIZED")
unk_ok = eval_results[5]["decision"] == "REJECTED"
print(f"\nKnown landmarks recognized: {known_ok}/5")
print(f"Unknown correctly rejected: {'Yes' if unk_ok else 'No'}")

Task 6: Performance Evaluation
Condition          Predicted              Keypoints    Matches    Decision       Time (ms)
----------------------------------------------------------------------------------------------------
  Original         Colosseum              91           91         RECOGNIZED     67.9
  Rotated 30deg    Colosseum              118          74         RECOGNIZED     67.6
  Resized 50%      Unknown                31           14         REJECTED       38.4
  Dark -40%        Colosseum              76           72         RECOGNIZED     54.7
  Diff Viewpoint   Colosseum              93           93         RECOGNIZED     61.6
  Unknown          N/A                    0            0          REJECTED       26.1

Known landmarks recognized: 4/5
Unknown correctly rejected: Yes


In [11]:
import matplotlib.pyplot as plt

OUT_DIR = "lab06_outputs"

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
cond_items = list(test_conds.items())

for i in range(6):
    ax = axes[i // 3, i % 3]
    cname, cimg = cond_items[i]
    r = eval_results[i]

    display = cv2.cvtColor(cimg, cv2.COLOR_BGR2RGB) if len(cimg.shape) == 3 else cimg

    # Green if correct behavior, red if wrong
    if i < 5:
        color = "green" if r["decision"] == "RECOGNIZED" else "red"
    else:
        color = "green" if r["decision"] == "REJECTED" else "red"

    ax.imshow(display)
    ax.set_title(f"{cname}\n{r['decision']}: {r['pred']}\n"
                 f"({r['kp']} kp, {r['matches']} matches)",
                 fontsize=10, color=color, fontweight="bold")
    ax.axis("off")

plt.suptitle("Task 6: Performance Under Different Conditions", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/07_performance.png", dpi=150, bbox_inches="tight")
plt.show()

In [12]:
import pandas as pd

OUT_DIR = "lab06_outputs"

df = pd.DataFrame(eval_results)
df.columns = ["Condition", "Predicted", "Keypoints", "Matches", "Decision", "Time (ms)"]
df.to_csv(f"{OUT_DIR}/evaluation_results.csv", index=False)
print("Results saved to evaluation_results.csv\n")
from IPython.display import display as ipy_display
ipy_display(df)

Results saved to evaluation_results.csv



,Condition,Predicted,Keypoints,Matches,Decision,Time (ms)
0,Original,Colosseum,91,91,RECOGNIZED,67.9
1,Rotated 30deg,Colosseum,118,74,RECOGNIZED,67.6
2,Resized 50%,Unknown,31,14,REJECTED,38.4
3,Dark -40%,Colosseum,76,72,RECOGNIZED,54.7
4,Diff Viewpoint,Colosseum,93,93,RECOGNIZED,61.6
5,Unknown,N/A,0,0,REJECTED,26.1


## 8. Multi-Landmark Recognition Test

In [13]:
import os, cv2
import matplotlib.pyplot as plt

DB_DIR = "landmark_database"
OUT_DIR = "lab06_outputs"
landmark_names = sorted([d for d in os.listdir(DB_DIR)
                         if os.path.isdir(os.path.join(DB_DIR, d))])
n_landmarks = len(landmark_names)

print("Testing recognition on each landmark...\n")

fig, axes = plt.subplots(2, n_landmarks, figsize=(4 * n_landmarks, 8))

correct = 0
for i, lm in enumerate(landmark_names):
    refs = database.get(lm, [])
    if len(refs) == 0:
        axes[0, i].set_title(lm, color="red")
        axes[0, i].axis("off")
        axes[1, i].axis("off")
        continue

    # Use last view as query
    qr = refs[-1]
    q_kp_i, q_des_i = sift.detectAndCompute(qr["gray"], None)

    if q_des_i is None:
        axes[0, i].set_title(lm + " (no features)", color="red")
        axes[0, i].axis("off")
        axes[1, i].axis("off")
        continue

    res = match_landmark(q_des_i, database, ratio=RATIO_THRESH)
    top_i = res[0]
    is_correct = top_i["landmark"] == lm
    if is_correct:
        correct += 1

    axes[0, i].imshow(cv2.cvtColor(qr["img"], cv2.COLOR_BGR2RGB))
    axes[0, i].set_title(f"Query: {lm.replace('_', ' ')}", fontsize=9, fontweight="bold")
    axes[0, i].axis("off")

    if top_i["ref"] is not None and len(top_i["good_matches"]) > 0:
        match_vis = cv2.drawMatches(
            qr["img"], q_kp_i,
            top_i["ref"]["img"], top_i["ref"]["kp"],
            top_i["good_matches"][:20], None,
            matchColor=(0, 255, 0),
            flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
        axes[1, i].imshow(cv2.cvtColor(match_vis, cv2.COLOR_BGR2RGB))

    tag = "OK" if is_correct else "WRONG"
    color = "green" if is_correct else "red"
    axes[1, i].set_title(f"Pred: {top_i['landmark'].replace('_', ' ')} "
                         f"({top_i['matches']} matches) [{tag}]",
                         fontsize=8, color=color, fontweight="bold")
    axes[1, i].axis("off")

plt.suptitle(f"Multi-Landmark Test: {correct}/{n_landmarks} correct", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/08_multi_test.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nAccuracy: {correct}/{n_landmarks} = {100*correct/max(n_landmarks,1):.1f}%")

Testing recognition on each landmark...


Accuracy: 5/5 = 100.0%


## 9. Test with Your Own Image (Optional)

Upload any landmark photo to test recognition.
**This cell is optional** -- it will not crash on Run All.

In [14]:
import cv2, time
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files as colab_files
    print("Upload a landmark image to test...")
    uploaded = colab_files.upload()
except Exception:
    uploaded = {}
    print("No file uploaded (skipping manual test).")

for fname, data in uploaded.items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        print(f"Could not read {fname}")
        continue

    img = cv2.resize(img, (400, 400))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    t0 = time.time()
    kp, des = sift.detectAndCompute(gray, None)

    if des is None or len(kp) < 2:
        print(f"Too few keypoints in {fname}")
        continue

    res = match_landmark(des, database, ratio=RATIO_THRESH)
    ms = (time.time() - t0) * 1000
    top_r = res[0]

    if top_r["matches"] >= MIN_MATCH_THRESHOLD:
        pred_text = top_r["landmark"].replace("_", " ")
        status = "RECOGNIZED"
    else:
        pred_text = "Not in database"
        status = "UNKNOWN"

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Your image: {fname}", fontsize=11)
    axes[0].axis("off")

    kp_vis = cv2.drawKeypoints(gray, kp, None, color=(0, 255, 0),
                                flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
    axes[1].imshow(cv2.cvtColor(kp_vis, cv2.COLOR_BGR2RGB))
    c = "green" if status == "RECOGNIZED" else "red"
    axes[1].set_title(f"{status}: {pred_text} ({top_r['matches']} matches, {ms:.0f}ms)",
                      fontsize=11, color=c, fontweight="bold")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    print(f"  Keypoints: {len(kp)} | Matches: {top_r['matches']} | "
          f"Prediction: {pred_text} | Time: {ms:.0f}ms")

Upload a landmark image to test...


Saving Screenshot 2026-10-16 114335.png to Screenshot 2026-10-16 114335.png
  Keypoints: 969 | Matches: 14 | Prediction: Not in database | Time: 379ms


## 10. Summary and Discussion

### How the System Works

1. **Database** -- 5 landmarks with 3 reference views each, downloaded from Wikimedia Commons
2. **SIFT Extraction** -- Each image yields hundreds of keypoints with 128-D descriptors
3. **BFMatcher + KNN** -- Finds 2 nearest neighbors for each query descriptor
4. **Lowe's Ratio Test** -- Keeps only matches where best is significantly better than second-best (ratio < 0.75)
5. **Recognition** -- Landmark with most reliable matches wins (if above threshold)
6. **Localization** -- Homography + RANSAC maps reference boundary onto query image

### Key Findings

- **Scale invariance**: SIFT detects features at multiple scales, so resized images still match
- **Rotation invariance**: Each keypoint's orientation is normalized, so rotated images work
- **Illumination robustness**: SIFT uses gradient orientations rather than raw intensity values
- **Unknown rejection**: The match threshold prevents false recognition of unseen landmarks
- **Lowe's ratio = 0.75**: Good balance between precision (fewer false matches) and recall (enough matches to recognize)

### Limitations

- SIFT is patented (use ORB or AKAZE for commercial use)
- Performance depends on image quality and viewpoint similarity
- No learning -- adding a new landmark requires new reference images
- Struggles with visually similar landmarks (e.g., similar buildings)

### Conclusion

SIFT-based landmark recognition is a robust, interpretable computer vision approach.
Its invariance to scale, rotation and illumination makes it well-suited for tourism
and navigation applications. For large-scale systems (thousands of landmarks),
deep learning (CNNs) would be more scalable.

In [15]:
import os, shutil

OUT_DIR = "lab06_outputs"
if os.path.exists("lab06_outputs.zip"):
    os.remove("lab06_outputs.zip")
shutil.make_archive("lab06_outputs", "zip", OUT_DIR)

print("All output images saved to lab06_outputs/")
print()
for f in sorted(os.listdir(OUT_DIR)):
    sz = os.path.getsize(os.path.join(OUT_DIR, f))
    print(f"  {f}  ({sz/1024:.1f} KB)")
print(f"\nTotal files: {len(os.listdir(OUT_DIR))}")

All output images saved to lab06_outputs/

  01_database.png  (4052.8 KB)
  02_sift_keypoints.png  (3416.5 KB)
  03_query_processing.png  (808.3 KB)
  04_matching.png  (699.7 KB)
  05_match_chart.png  (43.2 KB)
  06_localization.png  (746.3 KB)
  07_performance.png  (2477.1 KB)
  08_multi_test.png  (3431.7 KB)
  evaluation_results.csv  (0.3 KB)

Total files: 9
